# Emora — Emotion Model Training

Fine-tunes DistilBERT on 7-class emotion detection.

**Before running:**
1. Go to `Runtime → Change runtime type → T4 GPU` (free tier)
2. Click `Runtime → Run all`
3. At the end, a `emora_emotion_model.zip` file will download automatically
4. Unzip it into `backend/ml/models/transformer/` in your project

**Expected:** ~78-84% accuracy, ~5-8 minutes training time on T4 GPU.

In [ ]:
# ── Step 1: Verify GPU ────────────────────────────────────────────────────────
import torch
assert torch.cuda.is_available(), "No GPU found! Go to Runtime → Change runtime type → T4 GPU"
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ── Step 2: Install dependencies ─────────────────────────────────────────────
!pip install -q transformers==4.44.2 accelerate datasets scikit-learn pandas nltk

In [ ]:
# ── Step 3: Download GoEmotions dataset ──────────────────────────────────────
from datasets import load_dataset
import pandas as pd

EMOTION_MAPPING = {
    'admiration':'joy','amusement':'joy','approval':'joy','caring':'joy',
    'desire':'joy','excitement':'joy','gratitude':'joy','joy':'joy',
    'love':'joy','optimism':'joy','pride':'joy','relief':'joy',
    'anger':'anger','annoyance':'anger','disapproval':'anger',
    'disgust':'disgust',
    'embarrassment':'sadness','grief':'sadness','sadness':'sadness',
    'remorse':'sadness','disappointment':'sadness',
    'fear':'fear','nervousness':'fear',
    'confusion':'surprise','curiosity':'surprise',
    'realization':'surprise','surprise':'surprise',
    'neutral':'neutral',
}
CORE_EMOTIONS = sorted(['anger','disgust','fear','joy','neutral','sadness','surprise'])

print("Downloading GoEmotions...")
ds = load_dataset('go_emotions', 'simplified')

rows = []
for split in ['train', 'validation', 'test']:
    feat = ds[split].features['labels'].feature
    for item in ds[split]:
        if not item['labels']:
            continue
        label_28 = feat.int2str(item['labels'][0])
        label_7  = EMOTION_MAPPING.get(label_28)
        if label_7:
            rows.append({'text': item['text'], 'label': label_7, 'split': split})

df = pd.DataFrame(rows)
print(f"Total: {len(df):,} samples")
print(df['label'].value_counts())

In [ ]:
# ── Step 4: Prepare datasets ─────────────────────────────────────────────────
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset
from transformers import AutoTokenizer

MODEL_NAME = 'distilbert-base-uncased'
MAX_LEN    = 128
SEED       = 42

le = LabelEncoder()
le.fit(CORE_EMOTIONS)
df['label_id'] = le.transform(df['label'])

train_df = df[df['split'] == 'train']
val_df   = df[df['split'] == 'validation']
test_df  = df[df['split'] == 'test']

print(f"Train: {len(train_df):,}  Val: {len(val_df):,}  Test: {len(test_df):,}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class EmotionDataset(Dataset):
    def __init__(self, texts, labels):
        self.enc = tokenizer(
            list(texts), max_length=MAX_LEN,
            padding='max_length', truncation=True, return_tensors='pt'
        )
        self.labels = torch.tensor(list(labels), dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids':      self.enc['input_ids'][idx],
            'attention_mask': self.enc['attention_mask'][idx],
            'labels':         self.labels[idx],
        }

train_ds = EmotionDataset(train_df['text'], train_df['label_id'])
val_ds   = EmotionDataset(val_df['text'],   val_df['label_id'])
test_ds  = EmotionDataset(test_df['text'],  test_df['label_id'])
print("Datasets ready.")

In [ ]:
# ── Step 5: Build model ───────────────────────────────────────────────────────
from transformers import AutoModelForSequenceClassification

id2label = {i: l for i, l in enumerate(le.classes_)}
label2id = {l: i for i, l in id2label.items()}

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(CORE_EMOTIONS),
    id2label=id2label,
    label2id=label2id,
)
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# ── Step 6: Train ─────────────────────────────────────────────────────────────
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback
from sklearn.metrics import accuracy_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {'accuracy': accuracy_score(labels, preds)}

args = TrainingArguments(
    output_dir            = '/content/checkpoints',
    num_train_epochs      = 5,
    per_device_train_batch_size = 64,
    per_device_eval_batch_size  = 128,
    learning_rate         = 2e-5,
    warmup_ratio          = 0.1,
    weight_decay          = 0.01,
    eval_strategy         = 'epoch',
    save_strategy         = 'epoch',
    load_best_model_at_end= True,
    metric_for_best_model = 'accuracy',
    greater_is_better     = True,
    fp16                  = True,
    logging_steps         = 50,
    seed                  = SEED,
    report_to             = 'none',
)

trainer = Trainer(
    model           = model,
    args            = args,
    train_dataset   = train_ds,
    eval_dataset    = val_ds,
    compute_metrics = compute_metrics,
    callbacks       = [EarlyStoppingCallback(early_stopping_patience=2)],
)

print("Training... (~5-8 minutes on T4 GPU)")
trainer.train()

In [ ]:
# ── Step 7: Evaluate on test set ─────────────────────────────────────────────
from sklearn.metrics import classification_report

out   = trainer.predict(test_ds)
preds = np.argmax(out.predictions, axis=-1)
acc   = accuracy_score(test_df['label_id'], preds)

print(f"\n{'='*50}")
print(f"Test Accuracy: {acc:.4f}  ({acc*100:.1f}%)")
print(f"{'='*50}\n")
print(classification_report(
    test_df['label_id'], preds,
    target_names=le.classes_
))

In [ ]:
# ── Step 8: Save model + download zip ────────────────────────────────────────
import json, shutil, os
from google.colab import files

SAVE_DIR = '/content/emora_emotion_model'
os.makedirs(SAVE_DIR, exist_ok=True)

model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

meta = {
    'id2label':  {str(i): l for i, l in id2label.items()},
    'label2id':  label2id,
    'max_len':   MAX_LEN,
    'accuracy':  round(float(acc), 4),
    'source':    'fine-tuned distilbert-base-uncased on GoEmotions 7-class',
}
with open(f'{SAVE_DIR}/meta.json', 'w') as f:
    json.dump(meta, f, indent=2)

print(f"Accuracy achieved: {acc*100:.1f}%")
print("Zipping model...")
shutil.make_archive('/content/emora_emotion_model', 'zip', '/content', 'emora_emotion_model')

print("Downloading zip to your computer...")
files.download('/content/emora_emotion_model.zip')
print("\nDone! Now unzip into: backend/ml/models/transformer/")

## After downloading

1. Unzip `emora_emotion_model.zip`
2. Move the contents into `backend/ml/models/transformer/`
3. The folder should contain: `config.json`, `model.safetensors`, `tokenizer.json`, `meta.json`, etc.
4. Restart the backend — it will automatically detect and use the transformer model
5. You'll see in the logs: `transformer_model_loaded accuracy=0.XX`